# jev-my-bro — training on Google Colab

This notebook trains the active jev-my-bro v0.2 model on the Laya runtime. The repository already contains the static dataset; no dataset generation happens here.

Before running: Runtime → Change runtime type → GPU. Put the repository folder in Google Drive at `MyDrive/jev-my-bro` or change `SOURCE` below.

In [2]:
import shutil
import subprocess
import torch

if not torch.cuda.is_available():
    print('CUDA is unavailable in this Colab runtime.')
    print('Select Runtime -> Change runtime type -> T4 GPU, then reconnect and rerun this cell.')
    if shutil.which('nvidia-smi') is None:
        print('Diagnostic: nvidia-smi is not installed, so this is a CPU runtime.')
    raise RuntimeError('A Colab GPU runtime is required for RLCD fine-tuning.')

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi'], check=False, capture_output=True, text=True).stdout)
print('GPU:', torch.cuda.get_device_name(0))

/bin/bash: line 1: nvidia-smi: command not found


AssertionError: Enable a GPU runtime first

In [ ]:
from pathlib import Path
import os, shutil

CLI_SOURCE = Path('/content/jev-my-bro')
if (CLI_SOURCE / 'requirements.txt').exists():
    SOURCE = CLI_SOURCE
    REPO = Path('/content/jev-my-bro-run')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SOURCE = Path('/content/drive/MyDrive/jev-my-bro')
    REPO = Path('/content/jev-my-bro')
assert SOURCE.exists(), f'Upload the repo folder to {SOURCE} or edit SOURCE'
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SOURCE, REPO, ignore=shutil.ignore_patterns('artifacts', '__pycache__', '.git'))
os.chdir(REPO)
print('working directory:', Path.cwd())

In [ ]:
!pip install -q -r requirements.txt
import laya
print('laya:', laya.__version__)

In [ ]:
!python scripts/validate_dataset.py

## Train

The default base is `convaiinnovations/laya-multilingual` because this dataset contains English and Thai. The trainer is adapted for one Colab GPU using gradient accumulation.

In [ ]:
!python -m jevbro.train \
  --train data/train.jsonl \
  --validation data/validation.jsonl \
  --base-model convaiinnovations/laya-multilingual \
  --output artifacts/laya-model \
  --epochs 4 \
  --micro-batch 4 \
  --grad-accum 8 \
  --group-size 4

## Independent calibration

Only `data/calibration.jsonl` is used to fit temperatures. The test split remains untouched.

In [ ]:
!python -m jevbro.calibrate \
  --model artifacts/laya-model \
  --data data/calibration.jsonl \
  --report artifacts/calibration-report.json

## Final test

Run this after training and calibration. Do not tune the model against these results.

In [ ]:
!python -m jevbro.evaluate \
  --model artifacts/laya-model \
  --data data/test.jsonl \
  --device cuda \
  --report artifacts/test-report.json

In [ ]:
import laya, json
agent = laya.Agent('artifacts/laya-model', device='cuda')
from jevbro.questions import default_questions
samples = [
    'Agent wants to inspect git status without changing files',
    'เอเจนต์กำลังจะ deploy ระบบขึ้น production โดยยังไม่ได้รับอนุมัติ',
]
for text in samples:
    lang = 'th' if any('\u0e00' <= c <= '\u0e7f' for c in text) else 'en'
    result = agent.predict({'request': text}, default_questions(lang))
    print(text)
    print(json.dumps(result['answers'], ensure_ascii=False, indent=2))

## Save artifacts to Drive

In [ ]:
DEST = Path('/content/drive/MyDrive/jev-my-bro-artifacts')
if DEST.exists():
    shutil.rmtree(DEST)
shutil.copytree(REPO / 'artifacts', DEST)
print('saved:', DEST)